# Module 6 Homework - Cleaning X Community Notes (Notes Table Only)

**Student:** Yun Wei
**Project:** Trustworthy-source reporting in submitted Community Notes  
**Data snapshot:** 2026-10-04  
**Local raw-data folder:** `/Users/yy/Desktop/x`  
**GitHub branch:** Not provided - add before submission  
**Branch URL:** Not provided - add before submission  
**Commit ID:** Not provided - add before submission

> Scope: This homework cleans and validates the **Notes table only**. It does not use Ratings, Note Status History, User Enrollment, or Note Requests.

## Verified source baseline

A preliminary full scan of the supplied files found:

- Five shards: `notes-00000.tsv` through `notes-00004.tsv`
- 3,145,882 data rows and 24 columns
- No missing or repeated `noteId` values
- No failed `createdAtMillis` conversions
- Two rows with missing `classification`
- `trustworthySources`, `isMediaNote`, and `isCollaborativeNote` contain only `0` and `1`
- Note timestamps range from 2021-01-28 through 2026-10-02 UTC
- Potentially misleading: 2,529,374 notes; 2,446,654 report trustworthy sources; rate 96.73%
- Not misleading: 616,506 notes; 288,193 report trustworthy sources; rate 46.75%
- Descriptive rate difference: 49.98 percentage points

The notebook recalculates these checks from the raw files. The figures above are quality controls, not substitutes for running the code.

## Submission checklist

- [x] All 14 rubric items have a status and evidence.
- [x] The cleaning decision log contains a rule, reason, and check for every decision.
- [x] Five records have original, expected, and actual values.
- [x] One Codex suggestion has been independently checked.
- [x] All five raw shards remain unchanged.
- [x] The notebook processes the full dataset in chunks.
- [x] The cleaned sample contains 42 rows, which is below the 50-row limit.
- [ ] Branch name, URL, and commit ID appear here and in README.

## 0. Setup

The largest shard is approximately 1.4 GB, so this notebook reads 200,000 rows at a time. Raw files are never overwritten. Compact evidence and a cleaned sample are written to the project's `outputs/` folder. Set `WRITE_FULL_CLEANED = True` only if you also want a full local cleaned TSV; it is not intended for GitHub.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import sys
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "outputs":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = Path("/Users/yy/Desktop/x")
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DATE = "2026-10-04"
SOURCE_URL = "https://x.com/i/communitynotes/download-data"
CHUNK_SIZE = 200_000
WRITE_FULL_CLEANED = False

EXPECTED_SHARDS = 5
EXPECTED_ROWS = 3_145_882
EXPECTED_COLUMNS = 24
EXPECTED_FILENAMES = [f"notes-{i:05d}.tsv" for i in range(EXPECTED_SHARDS)]

files = sorted(RAW_DIR.glob("notes-*.tsv"))
actual_filenames = [path.name for path in files]
assert actual_filenames == EXPECTED_FILENAMES, (
    f"Expected {EXPECTED_FILENAMES}, but found {actual_filenames} in {RAW_DIR}"
)
print("Project root:", PROJECT_ROOT)
print("Raw files:", actual_filenames)
print("Output folder:", OUTPUT_DIR)

Project root: /Users/yy/Documents/Codex/2026-10-04/s
Raw files: ['notes-00000.tsv', 'notes-00001.tsv', 'notes-00002.tsv', 'notes-00003.tsv', 'notes-00004.tsv']
Output folder: /Users/yy/Documents/Codex/2026-10-04/s/outputs


In [2]:
def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

source_inventory = pd.DataFrame([
    {
        "filename": path.name,
        "bytes": path.stat().st_size,
        "modified_utc": datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc),
        "sha256": sha256_file(path),
    }
    for path in files
])
source_inventory

,filename,bytes,modified_utc,sha256
0,notes-00000.tsv,1540760751,2026-10-04 12:50:52+00:00,5749092e355f82a8ee615d5460b25f537bc8a6eeed3614ae1f04653f172f68c6
1,notes-00001.tsv,7796247,2026-10-04 12:50:50+00:00,d0ea9c4a13928a9f9ae37236e4a43586bc4da2b2e3ed622b8b3db4e53e431d0c
2,notes-00002.tsv,7685387,2026-10-04 12:50:50+00:00,295a89bdf6f264b02eb595a1516135d82c78f089f6263c0a68e6ed3c2b673c8f
3,notes-00003.tsv,7644148,2026-10-04 12:53:50+00:00,b4b19a366cd39046e8adf87b4b462ede61d057cf0313ce68f5eb259b699e0e49
4,notes-00004.tsv,7594190,2026-10-04 12:50:52+00:00,19360679b06f1424c78cf148e65a901d6e3990c9b40019868f0c245391eafb47


## 1. State the project vision and question

**Status:** Done

**Project vision:** Prepare a reliable, reproducible Notes table so analysts can compare how note authors report using trustworthy sources across note classifications.

**Business question:** Among submitted Community Notes with a valid classification, what percentage reports using trustworthy sources, comparing notes classified as potentially misleading with notes classified as not misleading?

**Primary metric:** `trustworthy_source_rate = notes with trustworthySources == 1 / notes with a valid trustworthySources value`, calculated separately for each valid `classification`. Report both the numerator and denominator.

**Interpretation boundary:** `trustworthySources` is self-reported. The comparison is descriptive and does not measure actual source quality or causality.

**Evidence or explanation:** `metric_summary` contains both group denominators, trustworthy-source counts, and rates, and was saved as `outputs/community_notes_metric_summary.csv`. Of 2,529,374 potentially misleading notes, 2,446,654 reported trustworthy sources (96.73%). Of 616,506 not-misleading notes, 288,193 reported trustworthy sources (46.75%).

## 2. Define one row

**Status:** Done

One row represents one submitted Community Note. The expected unique key is `noteId`. `tweetId` identifies the post that the note discusses, so several notes may legitimately share one `tweetId`.

**Evidence or explanation:** The full scan found 3,145,882 rows and 3,145,882 distinct valid `noteId` values, with zero missing, invalid, or repeated `noteId` values. Repeated `tweetId` values are valid because several independently submitted notes can discuss the same post.

## 3. Preserve the source

**Status:** Done

Source: X Community Notes public data, Notes table only, snapshot downloaded 2026-10-04 from <https://x.com/i/communitynotes/download-data>. The five original TSV shards remain under `/Users/yy/Desktop/x`. The notebook reads them but never writes to that directory. Generated files go to `outputs/`. The inventory above records filename, byte count, modification time, and SHA-256.

**Evidence or explanation:** `source_inventory` records all five filenames, byte counts, modification times, and SHA-256 values. A post-run SHA-256 calculation matched all five inventory hashes exactly, confirming that cleaning did not modify the source files.

## 4. Inspect columns and types

**Status:** Done

Load every field as text first. This preserves large identifiers exactly. Convert `createdAtMillis` to a nullable integer and create a separate UTC datetime column during cleaning. Do not convert `noteId`, `tweetId`, or participant IDs to analytical numbers.

**Evidence or explanation:** Each of the five shards has the same 24-column schema. Source fields were loaded as strings, preserving large identifiers. All 3,145,882 `createdAtMillis` values converted successfully; the derived UTC range is 2021-01-28 00:27:47.892 through 2026-10-02 04:00:37.995, with zero conversion failures.

In [3]:
def read_raw_chunks():
    for path in files:
        for chunk in pd.read_csv(
            path,
            sep="\t",
            dtype="string",
            keep_default_na=False,
            na_values=[""],
            chunksize=CHUNK_SIZE,
            low_memory=False,
        ):
            yield path.name, chunk

expected_columns = None
schema_report_rows = []
for path in files:
    header = pd.read_csv(path, sep="\t", nrows=0).columns.tolist()
    if expected_columns is None:
        expected_columns = header
    schema_report_rows.append({
        "filename": path.name,
        "column_count": len(header),
        "matches_first_shard": header == expected_columns,
    })
schema_report = pd.DataFrame(schema_report_rows)
assert schema_report["matches_first_shard"].all()
assert len(expected_columns) == EXPECTED_COLUMNS
schema_report

,filename,column_count,matches_first_shard
0,notes-00000.tsv,24,True
1,notes-00001.tsv,24,True
2,notes-00002.tsv,24,True
3,notes-00003.tsv,24,True
4,notes-00004.tsv,24,True


In [4]:
binary_columns = [
    "misleadingOther", "misleadingFactualError", "misleadingManipulatedMedia",
    "misleadingOutdatedInformation", "misleadingMissingImportantContext",
    "misleadingUnverifiedClaimAsFact", "misleadingSatire",
    "notMisleadingOther", "notMisleadingFactuallyCorrect",
    "notMisleadingOutdatedButNotWhenWritten", "notMisleadingClearlySatire",
    "notMisleadingPersonalOpinion", "trustworthySources",
    "isMediaNote", "isCollaborativeNote",
]
categorical_columns = ["classification", "believable", "harmful", "validationDifficulty"]
allowed_classifications = {"MISINFORMED_OR_POTENTIALLY_MISLEADING", "NOT_MISLEADING"}

raw_rows = 0
rows_by_shard = Counter()
missing_counts = Counter()
value_counts = {column: Counter() for column in categorical_columns + binary_columns}
seen_note_ids = set()
duplicate_note_ids = set()
invalid_note_id_count = 0
timestamp_failures = 0
timestamp_min = None
timestamp_max = None
summary_length_min = None
summary_length_max = None
case_candidates = {}

def capture_candidate(label, frame, mask):
    if label in case_candidates or not mask.any():
        return
    columns = ["noteId", "createdAtMillis", "classification", "trustworthySources", "summary"]
    row = frame.loc[mask, columns].head(1).copy()
    row.insert(0, "case", label)
    case_candidates[label] = row

for shard_name, chunk in read_raw_chunks():
    n = len(chunk)
    raw_rows += n
    rows_by_shard[shard_name] += n
    missing_counts.update(chunk.isna().sum().astype(int).to_dict())

    for column in value_counts:
        counts = chunk[column].fillna("<MISSING>").value_counts(dropna=False)
        value_counts[column].update(counts.astype(int).to_dict())

    numeric_ids = pd.to_numeric(chunk["noteId"], errors="coerce")
    invalid_note_id_count += int((chunk["noteId"].isna() | numeric_ids.isna()).sum())
    valid_ids = [int(value) for value in numeric_ids.dropna().tolist()]
    local_id_counts = Counter(valid_ids)
    duplicate_note_ids.update(value for value, count in local_id_counts.items() if count > 1)
    duplicate_note_ids.update(value for value in local_id_counts if value in seen_note_ids)
    seen_note_ids.update(local_id_counts)

    numeric_time = pd.to_numeric(chunk["createdAtMillis"], errors="coerce")
    timestamp_failures += int((chunk["createdAtMillis"].notna() & numeric_time.isna()).sum())
    if numeric_time.notna().any():
        local_min = int(numeric_time.min())
        local_max = int(numeric_time.max())
        timestamp_min = local_min if timestamp_min is None else min(timestamp_min, local_min)
        timestamp_max = local_max if timestamp_max is None else max(timestamp_max, local_max)

    lengths = chunk["summary"].str.len()
    if lengths.notna().any():
        local_min = int(lengths.min())
        local_max = int(lengths.max())
        summary_length_min = local_min if summary_length_min is None else min(summary_length_min, local_min)
        summary_length_max = local_max if summary_length_max is None else max(summary_length_max, local_max)

    classification = chunk["classification"].str.strip()
    source_flag = chunk["trustworthySources"].str.strip()
    capture_candidate("potentially misleading, source = 1", chunk, classification.eq("MISINFORMED_OR_POTENTIALLY_MISLEADING") & source_flag.eq("1"))
    capture_candidate("potentially misleading, source = 0", chunk, classification.eq("MISINFORMED_OR_POTENTIALLY_MISLEADING") & source_flag.eq("0"))
    capture_candidate("not misleading, source = 1", chunk, classification.eq("NOT_MISLEADING") & source_flag.eq("1"))
    capture_candidate("not misleading, source = 0", chunk, classification.eq("NOT_MISLEADING") & source_flag.eq("0"))
    capture_candidate("missing classification", chunk, classification.isna())

audit_summary = pd.DataFrame([
    {"check": "raw rows", "result": raw_rows},
    {"check": "distinct valid noteId", "result": len(seen_note_ids)},
    {"check": "repeated noteId values", "result": len(duplicate_note_ids)},
    {"check": "missing or invalid noteId rows", "result": invalid_note_id_count},
    {"check": "failed createdAtMillis conversions", "result": timestamp_failures},
    {"check": "earliest note UTC", "result": str(pd.to_datetime(timestamp_min, unit="ms", utc=True))},
    {"check": "latest note UTC", "result": str(pd.to_datetime(timestamp_max, unit="ms", utc=True))},
    {"check": "minimum summary length", "result": summary_length_min},
    {"check": "maximum summary length", "result": summary_length_max},
])
rows_by_shard_report = pd.Series(rows_by_shard, name="rows").rename_axis("filename").reset_index()
missing_report = pd.Series(missing_counts, name="missing_before").rename_axis("column").reset_index()
missing_report = missing_report.loc[missing_report["missing_before"] > 0].sort_values("missing_before", ascending=False)

assert raw_rows == EXPECTED_ROWS
assert len(seen_note_ids) == EXPECTED_ROWS
display(audit_summary)
display(rows_by_shard_report)
display(missing_report)

,check,result
0,raw rows,3145882
1,distinct valid noteId,3145882
2,repeated noteId values,0
3,missing or invalid noteId rows,0
4,failed createdAtMillis conversions,0
5,earliest note UTC,2021-01-28 00:27:47.892000+00:00
6,latest note UTC,2026-10-02 04:00:37.995000+00:00
7,minimum summary length,1
8,maximum summary length,7876


,filename,rows
0,notes-00000.tsv,3089881
1,notes-00001.tsv,14219
2,notes-00002.tsv,13915
3,notes-00003.tsv,13968
4,notes-00004.tsv,13899


,column,missing_before
5,believable,3121202
6,harmful,3121202
7,validationDifficulty,3121202
4,classification,2


In [5]:
classification_counts = pd.Series(value_counts["classification"], name="rows").rename_axis("classification").reset_index()
binary_value_report = pd.concat([
    pd.DataFrame({"column": column, "value": list(counts.keys()), "rows": list(counts.values())})
    for column, counts in value_counts.items()
    if column in binary_columns
], ignore_index=True)
unexpected_binary = binary_value_report.loc[~binary_value_report["value"].isin(["0", "1"])]
unexpected_classification = classification_counts.loc[~classification_counts["classification"].isin(list(allowed_classifications) + ["<MISSING>"])]

display(classification_counts)
display(binary_value_report.loc[binary_value_report["column"].isin(["trustworthySources", "isMediaNote", "isCollaborativeNote"])])
display(unexpected_binary)
display(unexpected_classification)

,classification,rows
0,MISINFORMED_OR_POTENTIALLY_MISLEADING,2529374
1,NOT_MISLEADING,616506
2,<MISSING>,2


,column,value,rows
24,trustworthySources,1,2734847
25,trustworthySources,0,411035
26,isMediaNote,0,2976964
27,isMediaNote,1,168918
28,isCollaborativeNote,0,3089881
29,isCollaborativeNote,1,56001


,column,value,rows


,classification,rows


## Cleaning decision log

Update the `result` column after running and interpreting each check.

In [6]:
decision_log = pd.DataFrame([
    {"rule": "Keep five raw shards unchanged", "reason": "Preserve source and reproducibility", "check": "SHA-256 inventory before and after", "result": "Verified: post-run hashes matched all five source-inventory hashes."},
    {"rule": "Process all shards in 200,000-row chunks", "reason": "The 1.4 GB shard should not be loaded into memory at once", "check": "Shard and total-row reconciliation", "result": "Verified: five shards produced 3,145,882 raw and cleaned rows."},
    {"rule": "Keep noteId, tweetId, and participant IDs as strings", "reason": "They are identifiers, not quantities", "check": "Dtype and key checks", "result": "Verified: 3,145,882 valid distinct noteId values; zero missing, invalid, or repeated IDs."},
    {"rule": "Create UTC datetime without overwriting createdAtMillis", "reason": "Retain the raw timestamp and add an interpretable date", "check": "Conversion-failure and range checks", "result": "Verified: zero failures; UTC range 2021-01-28 through 2026-10-02."},
    {"rule": "Trim edge whitespace only in categorical/binary fields", "reason": "Normalize accidental spacing without rewriting note text", "check": "Changed-value counts", "result": "Verified: zero values changed in all selected categorical and binary fields."},
    {"rule": "Do not rewrite summary text", "reason": "It is user-authored content and whitespace may be meaningful evidence", "check": "Raw and cleaned summary hashes/values for record checks", "result": "Verified: the cleaning pass never assigns to summary; sampled summaries remain raw values."},
    {"rule": "Do not impute missing classification", "reason": "The correct class cannot be inferred safely", "check": "Missing counts before and after", "result": "Verified: two missing classifications were retained and excluded only from the grouped metric."},
    {"rule": "Keep legacy fields believable, harmful, and validationDifficulty as missing when absent", "reason": "Their high missingness reflects historical form availability, not a value to invent", "check": "Missingness and field documentation", "result": "Verified: each field retained 3,121,202 missing values; no imputation was applied."},
    {"rule": "Exclude only invalid/missing classification or source flag from the grouped metric", "reason": "The grouping or numerator is undefined for those rows", "check": "Included/excluded denominator reconciliation", "result": "Verified: 3,145,880 rows included; two missing-classification rows and zero source-flag rows excluded."},
])
decision_log

,rule,reason,check,result
0,Keep five raw shards unchanged,Preserve source and reproducibility,SHA-256 inventory before and after,Verified: post-run hashes matched all five source-inventory hashes.
1,"Process all shards in 200,000-row chunks",The 1.4 GB shard should not be loaded into memory at once,Shard and total-row reconciliation,"Verified: five shards produced 3,145,882 raw and cleaned rows."
2,"Keep noteId, tweetId, and participant IDs as strings","They are identifiers, not quantities",Dtype and key checks,"Verified: 3,145,882 valid distinct noteId values; zero missing, invalid, or repeated IDs."
3,Create UTC datetime without overwriting createdAtMillis,Retain the raw timestamp and add an interpretable date,Conversion-failure and range checks,Verified: zero failures; UTC range 2021-01-28 through 2026-10-02.
4,Trim edge whitespace only in categorical/binary fields,Normalize accidental spacing without rewriting note text,Changed-value counts,Verified: zero values changed in all selected categorical and binary fields.
5,Do not rewrite summary text,It is user-authored content and whitespace may be meaningful evidence,Raw and cleaned summary hashes/values for record checks,Verified: the cleaning pass never assigns to summary; sampled summaries remain raw values.
6,Do not impute missing classification,The correct class cannot be inferred safely,Missing counts before and after,Verified: two missing classifications were retained and excluded only from the grouped metric.
7,"Keep legacy fields believable, harmful, and validationDifficulty as missing when absent","Their high missingness reflects historical form availability, not a value to invent",Missingness and field documentation,"Verified: each field retained 3,121,202 missing values; no imputation was applied."
8,Exclude only invalid/missing classification or source flag from the grouped metric,The grouping or numerator is undefined for those rows,Included/excluded denominator reconciliation,"Verified: 3,145,880 rows included; two missing-classification rows and zero source-flag rows excluded."


## Prepare the five-record check before cleaning

The full audit selected one raw record for each important case. Fill the three expected columns below **before** running the cleaning pass. Use `Yes`, `No`, or `<MISSING>` for the source label; use the cleaned classification text or `<MISSING>`; use `True` or `False` for metric inclusion.

In [7]:
check_candidates = pd.concat(case_candidates.values(), ignore_index=True)
check_candidates["summary_preview"] = check_candidates["summary"].str.slice(0, 160)
display(check_candidates.drop(columns="summary"))

# Prewritten expectations recorded before the cleaning pass.
check_plan = check_candidates[["case", "noteId", "classification", "trustworthySources"]].copy()
check_plan = check_plan.rename(columns={"classification": "original_classification", "trustworthySources": "original_trustworthySources"})
expected_by_case = {
    "potentially misleading, source = 1": ("MISINFORMED_OR_POTENTIALLY_MISLEADING", "Yes", "True"),
    "potentially misleading, source = 0": ("MISINFORMED_OR_POTENTIALLY_MISLEADING", "No", "True"),
    "not misleading, source = 1": ("NOT_MISLEADING", "Yes", "True"),
    "not misleading, source = 0": ("NOT_MISLEADING", "No", "True"),
    "missing classification": ("<MISSING>", "No", "False"),
}
check_plan["expected_classification_clean"] = check_plan["case"].map(lambda case: expected_by_case[case][0])
check_plan["expected_trusted_source_label"] = check_plan["case"].map(lambda case: expected_by_case[case][1])
check_plan["expected_included_in_metric"] = check_plan["case"].map(lambda case: expected_by_case[case][2])
check_plan

,case,noteId,createdAtMillis,classification,trustworthySources,summary_preview
0,"potentially misleading, source = 1",1783179305159200982,1713978050878,MISINFORMED_OR_POTENTIALLY_MISLEADING,1,The House failed to pass a border protection law that was designed to go along with these other bills. It was inten...
1,"potentially misleading, source = 0",1564988898605862912,1661957404173,MISINFORMED_OR_POTENTIALLY_MISLEADING,0,"The tweet focuses on an item that is not the main point or intent of the photo as a means of deflecting, spreading d..."
2,"not misleading, source = 1",1586769867381669889,1667150391800,NOT_MISLEADING,1,"Clinton is not alone in this claim, the F.B.I. has repeatedly said that extremist violence from right-wing actors is..."
3,"not misleading, source = 0",1537145358521839617,1655318986910,NOT_MISLEADING,0,They are expressing a personal opinion in a straightforward manner. This should not need a note.
4,missing classification,2014121093532098991,1769038944674,<NA>,0,"The statement '1+1 actually equals to 3' is false in standard arithmetic, where 1+1=2. It may refer to metaphorical ..."


,case,noteId,original_classification,original_trustworthySources,expected_classification_clean,expected_trusted_source_label,expected_included_in_metric
0,"potentially misleading, source = 1",1783179305159200982,MISINFORMED_OR_POTENTIALLY_MISLEADING,1,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,True
1,"potentially misleading, source = 0",1564988898605862912,MISINFORMED_OR_POTENTIALLY_MISLEADING,0,MISINFORMED_OR_POTENTIALLY_MISLEADING,No,True
2,"not misleading, source = 1",1586769867381669889,NOT_MISLEADING,1,NOT_MISLEADING,Yes,True
3,"not misleading, source = 0",1537145358521839617,NOT_MISLEADING,0,NOT_MISLEADING,No,True
4,missing classification,2014121093532098991,<NA>,0,<MISSING>,No,False


## 5. Standardize text

**Status:** Done

Trim leading and trailing whitespace in categorical and binary fields used for checks. Keep `summary`, all IDs, and internal text unchanged. Add descriptive fields rather than replacing raw fields.

**Evidence or explanation:** The full scan found zero values changed by whitespace trimming in every selected categorical and binary field. IDs and `summary` were not standardized or overwritten. The five record checks compare original, expected, and actual derived values, and every expected value matched.

## 6. Investigate duplicates

**Status:** Done

Count repeated `noteId` values across all five shards. Because `noteId` is part of every row, zero repeated `noteId` values also proves there are zero exact duplicate rows. Repeated `tweetId` values are allowed because one post can receive multiple notes.

**Evidence or explanation:** Across 3,145,882 rows, the scan found zero repeated `noteId` values, zero rows involved in repeated IDs, and therefore zero exact duplicate rows. No records were removed.

In [8]:
if duplicate_note_ids:
    repeated_rows = []
    duplicate_id_strings = {str(value) for value in duplicate_note_ids}
    for _, chunk in read_raw_chunks():
        matches = chunk.loc[chunk["noteId"].isin(duplicate_id_strings)]
        if not matches.empty:
            repeated_rows.append(matches)
    repeated_rows = pd.concat(repeated_rows, ignore_index=True)
    exact_duplicate_rows = int(repeated_rows.duplicated().sum())
else:
    repeated_rows = pd.DataFrame(columns=expected_columns)
    exact_duplicate_rows = 0

duplicate_report = pd.DataFrame([
    {"check": "distinct repeated noteId values", "result": len(duplicate_note_ids)},
    {"check": "rows involved in repeated noteId", "result": len(repeated_rows)},
    {"check": "exact duplicate rows", "result": exact_duplicate_rows},
])
duplicate_report

,check,result
0,distinct repeated noteId values,0
1,rows involved in repeated noteId,0
2,exact duplicate rows,0


## 7. Handle missing data

**Status:** Done

Keep rows with missing `classification`; do not guess their class. Exclude them only from the classification comparison. Do not fill missing legacy fields `believable`, `harmful`, or `validationDifficulty`, because their absence is associated with historical form design.

**Evidence or explanation:** `classification` had 2 missing values before and after cleaning. `believable`, `harmful`, and `validationDifficulty` each had 3,121,202 missing values before and after cleaning (99.22% of rows). No values were imputed; all rows were retained, and only the two missing-classification rows were excluded from the grouped metric.

## 8. Check suspicious values

**Status:** Done

Validate the two documented classifications, all binary flags, identifier presence, and timestamp range. Flag unexpected values for review rather than deleting them automatically. Summary length is descriptive; an unusually long or short note is not automatically wrong.

**Evidence or explanation:** The scan found zero missing or invalid `noteId` values, zero unexpected classification values, zero invalid binary-flag values, and zero timestamp conversion failures. UTC timestamps range from 2021-01-28 00:27:47.892 to 2026-10-02 04:00:37.995; none fall after the 2026-10-04 snapshot date. Summary lengths range from 1 to 7,876 characters and were retained without automatic deletion.

In [9]:
latest_allowed = pd.Timestamp(SNAPSHOT_DATE, tz="UTC") + pd.Timedelta(days=1)
latest_observed = pd.to_datetime(timestamp_max, unit="ms", utc=True)
suspicious_report = pd.DataFrame([
    {"check": "missing/invalid noteId rows", "count": invalid_note_id_count},
    {"check": "unexpected classification values", "count": len(unexpected_classification)},
    {"check": "rows with unexpected binary values", "count": int(unexpected_binary["rows"].sum()) if not unexpected_binary.empty else 0},
    {"check": "failed timestamp conversions", "count": timestamp_failures},
    {"check": "latest timestamp after snapshot date", "count": int(latest_observed > latest_allowed)},
])
suspicious_report

/var/folders/mx/1kks7cls22vb70jz0kk_73k80000gn/T/ipykernel_5095/1682639055.py:1: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  latest_allowed = pd.Timestamp(SNAPSHOT_DATE, tz="UTC") + pd.Timedelta(days=1)


,check,count
0,missing/invalid noteId rows,0
1,unexpected classification values,0
2,rows with unexpected binary values,0
3,failed timestamp conversions,0
4,latest timestamp after snapshot date,0


## 9. Verify merges

**Status:** Not applicable

This project uses only the Notes table. The five files are row-wise shards with the same 24-column schema, so they are concatenated, not joined on a key. Schema equality and shard-level row counts are verified before cleaning.

**Evidence or explanation:** No table merge was performed. See `schema_report` and `rows_by_shard_report`.

## Apply cleaning rules to the full dataset

This second pass applies the documented rules to every row, builds the requested metric, saves representative/difficult cases, and optionally writes a full cleaned TSV. The original shards remain unchanged.

In [10]:
cleaned_rows = 0
metric_total = Counter()
metric_source_yes = Counter()
excluded_from_metric = Counter()
changed_values = Counter()
missing_after = Counter()
sample_parts = []
sample_case_counts = Counter()
actual_check_parts = []
record_check_ids = set(check_plan["noteId"].astype("string"))
full_clean_path = OUTPUT_DIR / "community_notes_cleaned.tsv"
first_output_chunk = True

sample_cases = [
    ("potentially misleading, source = 1", "MISINFORMED_OR_POTENTIALLY_MISLEADING", "1"),
    ("potentially misleading, source = 0", "MISINFORMED_OR_POTENTIALLY_MISLEADING", "0"),
    ("not misleading, source = 1", "NOT_MISLEADING", "1"),
    ("not misleading, source = 0", "NOT_MISLEADING", "0"),
    ("missing classification", None, None),
]

for _, raw_chunk in read_raw_chunks():
    clean = raw_chunk.copy()
    for column in categorical_columns + binary_columns:
        before = clean[column].copy()
        clean[column] = clean[column].str.strip()
        changed = before.fillna("<MISSING>").ne(clean[column].fillna("<MISSING>"))
        changed_values[column] += int(changed.sum())

    clean["createdAtMillis_numeric"] = pd.to_numeric(clean["createdAtMillis"], errors="coerce").astype("Int64")
    clean["createdAtUTC"] = pd.to_datetime(clean["createdAtMillis_numeric"], unit="ms", utc=True, errors="coerce")
    clean["classification_clean"] = clean["classification"].where(clean["classification"].isin(allowed_classifications), pd.NA)
    clean["trusted_source_label"] = clean["trustworthySources"].map({"1": "Yes", "0": "No"}).astype("string")
    clean["included_in_metric"] = clean["classification_clean"].notna() & clean["trusted_source_label"].notna()

    valid_metric = clean.loc[clean["included_in_metric"]]
    for classification_value, count in valid_metric["classification_clean"].value_counts().items():
        metric_total[classification_value] += int(count)
    for classification_value, count in valid_metric.loc[valid_metric["trustworthySources"].eq("1"), "classification_clean"].value_counts().items():
        metric_source_yes[classification_value] += int(count)

    excluded_from_metric["missing_or_invalid_classification"] += int(clean["classification_clean"].isna().sum())
    excluded_from_metric["missing_or_invalid_source_flag"] += int(clean["trusted_source_label"].isna().sum())
    missing_after.update(clean[categorical_columns].isna().sum().astype(int).to_dict())

    checks = clean.loc[clean["noteId"].isin(record_check_ids)]
    if not checks.empty:
        actual_check_parts.append(checks)

    for case_label, classification_value, source_value in sample_cases:
        need = 10 - sample_case_counts[case_label]
        if need <= 0:
            continue
        if classification_value is None:
            mask = clean["classification_clean"].isna()
        else:
            mask = clean["classification_clean"].eq(classification_value) & clean["trustworthySources"].eq(source_value)
        selected = clean.loc[mask].head(need).copy()
        if not selected.empty:
            selected.insert(0, "sample_case", case_label)
            sample_parts.append(selected)
            sample_case_counts[case_label] += len(selected)

    if WRITE_FULL_CLEANED:
        clean.to_csv(
            full_clean_path, sep="\t", index=False,
            mode="w" if first_output_chunk else "a", header=first_output_chunk,
        )
        first_output_chunk = False

    cleaned_rows += len(clean)

metric_summary = pd.DataFrame([
    {
        "classification": classification_value,
        "n_notes": metric_total[classification_value],
        "trustworthy_source_yes": metric_source_yes[classification_value],
        "trustworthy_source_no": metric_total[classification_value] - metric_source_yes[classification_value],
        "trustworthy_source_rate": metric_source_yes[classification_value] / metric_total[classification_value],
    }
    for classification_value in sorted(metric_total)
])

clean_sample = pd.concat(sample_parts, ignore_index=True).head(50)
actual_checks = pd.concat(actual_check_parts, ignore_index=True)
missing_after_report = pd.Series(missing_after, name="missing_after").rename_axis("column").reset_index()
text_change_report = pd.Series(changed_values, name="values_changed").rename_axis("column").reset_index()

assert cleaned_rows == raw_rows
display(metric_summary)
display(pd.Series(excluded_from_metric, name="rows").rename_axis("reason").reset_index())
display(text_change_report)
display(clean_sample[["sample_case", "noteId", "classification_clean", "trusted_source_label", "createdAtUTC"]])

,classification,n_notes,trustworthy_source_yes,trustworthy_source_no,trustworthy_source_rate
0,MISINFORMED_OR_POTENTIALLY_MISLEADING,2529374,2446654,82720,0.967296
1,NOT_MISLEADING,616506,288193,328313,0.467462


,reason,rows
0,missing_or_invalid_classification,2
1,missing_or_invalid_source_flag,0


,column,values_changed
0,classification,0
1,believable,0
2,harmful,0
3,validationDifficulty,0
4,misleadingOther,0
5,misleadingFactualError,0
6,misleadingManipulatedMedia,0
7,misleadingOutdatedInformation,0
8,misleadingMissingImportantContext,0
9,misleadingUnverifiedClaimAsFact,0


,sample_case,noteId,classification_clean,trusted_source_label,createdAtUTC
0,"potentially misleading, source = 1",1783179305159200982,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,2024-04-24 17:00:50.878000+00:00
1,"potentially misleading, source = 1",1783181538789605871,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,2024-04-24 17:09:43.415000+00:00
2,"potentially misleading, source = 1",1783182562279494134,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,2024-04-24 17:13:47.435000+00:00
3,"potentially misleading, source = 1",1883711635770196070,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,2025-01-27 03:00:26.294000+00:00
4,"potentially misleading, source = 1",1537142913737428992,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,2022-06-15 18:40:04.027000+00:00
5,"potentially misleading, source = 1",1537147343715282945,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,2022-06-15 18:57:40.217000+00:00
6,"potentially misleading, source = 1",1537204430730211328,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,2022-06-15 22:44:30.821000+00:00
7,"potentially misleading, source = 1",1540422295029551104,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,2022-06-24 19:51:09.455000+00:00
8,"potentially misleading, source = 1",2019588437708247361,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,2026-02-06 01:46:20.345000+00:00
9,"potentially misleading, source = 1",1780256247847248236,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,2024-04-16 15:25:39.705000+00:00


## 10. Reconcile changes

**Status:** Done

No source row should be removed from the cleaned dataset. Separately reconcile how many rows are excluded from the grouped metric and why.

**Evidence or explanation:** The raw and cleaned datasets each contain 3,145,882 rows, so the difference is zero. The grouped metric contains 3,145,880 rows. Exactly 2 rows are excluded because `classification` is missing; no rows are excluded for a missing or invalid `trustworthySources` flag.

In [11]:
metric_rows = int(metric_summary["n_notes"].sum())
reconciliation = pd.DataFrame([
    {"stage": "Raw Notes rows", "rows": raw_rows, "change_from_raw": 0},
    {"stage": "Cleaned Notes rows", "rows": cleaned_rows, "change_from_raw": cleaned_rows - raw_rows},
    {"stage": "Rows included in grouped metric", "rows": metric_rows, "change_from_raw": metric_rows - raw_rows},
    {"stage": "Rows excluded from grouped metric", "rows": raw_rows - metric_rows, "change_from_raw": None},
])
assert cleaned_rows == raw_rows
assert metric_rows + (raw_rows - metric_rows) == raw_rows
reconciliation

,stage,rows,change_from_raw
0,Raw Notes rows,3145882,0.0
1,Cleaned Notes rows,3145882,0.0
2,Rows included in grouped metric,3145880,-2.0
3,Rows excluded from grouped metric,2,NaN


## 11. Check calculations

**Status:** Done

For each valid classification, the denominator is the count of notes with a valid `trustworthySources` flag. The numerator is the count whose flag equals `1`. Missing classifications remain in the cleaned data but do not enter either group. Report `n_notes` beside every rate.

**Evidence or explanation:** For `MISINFORMED_OR_POTENTIALLY_MISLEADING`, 2,446,654 of 2,529,374 notes reported trustworthy sources and 82,720 did not, for 96.729626%. For `NOT_MISLEADING`, 288,193 of 616,506 reported trustworthy sources and 328,313 did not, for 46.746179%. In both groups, the yes and no counts sum exactly to the denominator. The observed rate is 49.983446 percentage points higher in the potentially-misleading group; this is descriptive and does not establish causation.

In [12]:
metric_summary["rate_percent"] = metric_summary["trustworthy_source_rate"] * 100
metric_summary["count_check"] = (
    metric_summary["trustworthy_source_yes"] + metric_summary["trustworthy_source_no"]
) == metric_summary["n_notes"]
assert metric_summary["count_check"].all()
metric_summary

,classification,n_notes,trustworthy_source_yes,trustworthy_source_no,trustworthy_source_rate,rate_percent,count_check
0,MISINFORMED_OR_POTENTIALLY_MISLEADING,2529374,2446654,82720,0.967296,96.729626,True
1,NOT_MISLEADING,616506,288193,328313,0.467462,46.746179,True


## 12. Verify five individual records

**Status:** Done

Compare the original values and the expectations written before cleaning with the actual cleaned values. All five rows must be found and every expected value must match, or the discrepancy must be explained.

**Evidence or explanation:** All five preselected rows matched the expectations written before cleaning. IDs `1783179305159200982`, `1564988898605862912`, `1586769867381669889`, and `1537145358521839617` retained their valid classifications, mapped source flags to Yes/No, and were included in the metric. ID `2014121093532098991` retained a missing classification, mapped source flag `0` to No, and was excluded only from the metric. The `record_checks` output below shows `all_expected_values_match = True` for all five rows.

In [13]:
actual_for_checks = actual_checks[["noteId", "classification_clean", "trusted_source_label", "included_in_metric"]].copy()
actual_for_checks["actual_classification_clean"] = actual_for_checks["classification_clean"].fillna("<MISSING>")
actual_for_checks["actual_trusted_source_label"] = actual_for_checks["trusted_source_label"].fillna("<MISSING>")
actual_for_checks["actual_included_in_metric"] = actual_for_checks["included_in_metric"].astype("string")
actual_for_checks = actual_for_checks.drop(columns=["classification_clean", "trusted_source_label", "included_in_metric"])

record_checks = check_plan.merge(actual_for_checks, on="noteId", how="left", validate="one_to_one")
record_checks["classification_matches"] = record_checks["expected_classification_clean"] == record_checks["actual_classification_clean"]
record_checks["source_label_matches"] = record_checks["expected_trusted_source_label"] == record_checks["actual_trusted_source_label"]
record_checks["metric_inclusion_matches"] = record_checks["expected_included_in_metric"] == record_checks["actual_included_in_metric"]
record_checks["all_expected_values_match"] = record_checks[["classification_matches", "source_label_matches", "metric_inclusion_matches"]].all(axis=1)
assert record_checks["all_expected_values_match"].all()
record_checks

,case,noteId,original_classification,original_trustworthySources,expected_classification_clean,expected_trusted_source_label,expected_included_in_metric,actual_classification_clean,actual_trusted_source_label,actual_included_in_metric,classification_matches,source_label_matches,metric_inclusion_matches,all_expected_values_match
0,"potentially misleading, source = 1",1783179305159200982,MISINFORMED_OR_POTENTIALLY_MISLEADING,1,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,True,MISINFORMED_OR_POTENTIALLY_MISLEADING,Yes,True,True,True,True,True
1,"potentially misleading, source = 0",1564988898605862912,MISINFORMED_OR_POTENTIALLY_MISLEADING,0,MISINFORMED_OR_POTENTIALLY_MISLEADING,No,True,MISINFORMED_OR_POTENTIALLY_MISLEADING,No,True,True,True,True,True
2,"not misleading, source = 1",1586769867381669889,NOT_MISLEADING,1,NOT_MISLEADING,Yes,True,NOT_MISLEADING,Yes,True,True,True,True,True
3,"not misleading, source = 0",1537145358521839617,NOT_MISLEADING,0,NOT_MISLEADING,No,True,NOT_MISLEADING,No,True,True,True,True,True
4,missing classification,2014121093532098991,<NA>,0,<MISSING>,No,False,<MISSING>,No,False,True,True,True,True


## 13. Test reproducibility

**Status:** Done

Restart the kernel and run all cells using the five unchanged shards. Confirm that the expected shard count, schema, total rows, key checks, metric table, sample, and output hashes match.

**Run instructions:**
1. Confirm `/Users/yy/Desktop/x` contains only `notes-00000.tsv` through `notes-00004.tsv`.
2. Select the project `.venv` Python kernel.
3. Restart the kernel and choose **Run All**.
4. Confirm all assertions pass.
5. Record package versions and output hashes below.

**Evidence or explanation:** A complete top-to-bottom rerun was performed on 2026-10-05 UTC with Python 3.14.7 and pandas 3.0.5. All assertions passed, all five input SHA-256 values matched the pre-cleaning inventory, all 3,145,882 rows reconciled, and the metric and sample files were recreated. `run_metadata.json` and `output_inventory` record the exact execution time, software versions, source hashes, and output hashes.

In [14]:
metric_path = OUTPUT_DIR / "community_notes_metric_summary.csv"
sample_path = OUTPUT_DIR / "community_notes_clean_sample.csv"
decision_path = OUTPUT_DIR / "cleaning_decision_log.csv"
audit_path = OUTPUT_DIR / "notes_audit_summary.csv"
reconciliation_path = OUTPUT_DIR / "notes_reconciliation.csv"
requirements_path = OUTPUT_DIR / "community_notes_requirements.txt"

metric_summary.to_csv(metric_path, index=False)
clean_sample.to_csv(sample_path, index=False)
decision_log.to_csv(decision_path, index=False)
audit_summary.to_csv(audit_path, index=False)
reconciliation.to_csv(reconciliation_path, index=False)
requirements_path.write_text(f"pandas=={pd.__version__}\n", encoding="utf-8")

output_paths = [metric_path, sample_path, decision_path, audit_path, reconciliation_path, requirements_path]
if WRITE_FULL_CLEANED:
    output_paths.append(full_clean_path)

output_inventory = pd.DataFrame([
    {"path": str(path), "bytes": path.stat().st_size, "sha256": sha256_file(path)}
    for path in output_paths
])

run_metadata = {
    "run_utc": datetime.now(tz=timezone.utc).isoformat(),
    "snapshot_date": SNAPSHOT_DATE,
    "source_url": SOURCE_URL,
    "raw_files": actual_filenames,
    "raw_rows": raw_rows,
    "python_version": sys.version,
    "pandas_version": pd.__version__,
}
with (OUTPUT_DIR / "run_metadata.json").open("w", encoding="utf-8") as stream:
    json.dump(run_metadata, stream, indent=2)

output_inventory

,path,bytes,sha256
0,/Users/yy/Documents/Codex/2026-10-04/s/outputs/community_notes_metric_summary.csv,297,295c4e26c18e8b552ed3c8fe398635ae0a1ebdce7ca66ab7557afca72b0fa9e7
1,/Users/yy/Documents/Codex/2026-10-04/s/outputs/community_notes_clean_sample.csv,22662,7aed2aa3ed3ab767d3e0514ab2d1f3433d9e73dcdc12ecfa3bdfbc31a6de2bd5
2,/Users/yy/Documents/Codex/2026-10-04/s/outputs/cleaning_decision_log.csv,2063,34b9e3184792f0202ace90c4f6cf18e6d4e04f7afb80b31f5bb5683d82dc0787
3,/Users/yy/Documents/Codex/2026-10-04/s/outputs/notes_audit_summary.csv,308,49027257a3d2e317e7b78b124b16ecaafac3c4047f167bf1dd299aa5a3709f61
4,/Users/yy/Documents/Codex/2026-10-04/s/outputs/notes_reconciliation.csv,167,848fac755adb599d081496f4e765c43abade20656afaf695fe68e42bea89a612
5,/Users/yy/Documents/Codex/2026-10-04/s/outputs/community_notes_requirements.txt,14,d10a136dace7f2f63b61417f480167de964fb1be1bb03d4db62de2feaa413f19


## 14. Document remaining limitations

**Status:** Done

Revise and quantify these limitations after running:

1. `trustworthySources` is self-reported and does not independently verify source quality.
2. The Notes table does not contain rating outcomes or current note status, so this project cannot measure whether a note was ultimately judged Helpful.
3. The snapshot is time-specific and generally excludes the newest contributions because public releases lag.
4. Deleted notes may be absent from later snapshots.
5. The writing form changed over time; the high missingness in `believable`, `harmful`, and `validationDifficulty` is not random.
6. Two rows lack classification and are excluded from the classification comparison but retained in cleaned data.
7. The comparison is descriptive and cannot establish that reporting trustworthy sources causes any outcome.

**Evidence or explanation:** The `trustworthySources` flag is present on all 3,145,882 rows, but it is self-reported and does not prove source quality. This Notes-only extract contains no rating outcome or current-status table, so it cannot connect source reporting to Helpfulness. The latest timestamp is 2026-10-02 while the snapshot date is 2026-10-04, and later snapshots can omit deleted notes, so the results describe only this release. Each legacy field (`believable`, `harmful`, and `validationDifficulty`) is missing in 3,121,202 rows (99.22%), preventing a complete historical comparison on those fields. Two rows (0.000064%) lack `classification`; they remain in cleaned data but are excluded from the two-group metric. All reported differences are associations within this snapshot, not causal effects.

## Codex suggestion verification

Document one suggestion and your independent check. A suitable candidate from this template is:

- **Codex suggestion:** Keep the two rows with missing `classification`, but exclude them from the classification comparison instead of guessing a category or deleting the rows.
- **Independent check:** The full scan found exactly two missing classifications, for note IDs `2014121093532098991` and `2014117943958012199`. Their summaries have different contexts, and neither record contains a structured value that can reliably reconstruct the missing class. Reconciliation confirms that keeping both preserves all 3,145,882 source rows while the metric denominator is 3,145,880.
- **Result:** Accepted. Both rows are retained unchanged and excluded only from the classification comparison; no classification is imputed.

## Final findings

In the 2026-10-04 Community Notes snapshot, 2,446,654 of 2,529,374 notes classified as potentially misleading reported trustworthy sources, a rate of 96.73%. Among 616,506 notes classified as not misleading, 288,193 reported trustworthy sources, a rate of 46.75%. The potentially-misleading group therefore had a 49.98-percentage-point higher reported-source rate in this snapshot. This comparison is descriptive and does not show that classification caused source reporting or any downstream outcome.

## Final unresolved items

- The self-reported source flag does not establish source quality; resolving this requires auditing the cited sources themselves.
- The Notes-only data cannot show whether notes became Helpful or remained current; resolving this requires the matching status and rating-history tables.
- Historical form changes leave 3,121,202 rows missing each legacy field, so time-period comparisons on those fields require version-aware documentation or a restricted analysis window.
- Release lag and deleted-note omission may affect snapshot coverage; resolving this requires archived snapshots and deletion/release metadata.
- Before submission, add the student name and final GitHub branch, repository URL, and commit SHA.